# Blind Eval From OSZ Glob

This notebook is a quick entry workflow for blind evaluation. It reuses the existing OSZ-selection logic from `infer_models_from_osz_glob.ipynb`, but drives generation through the same model registry and inference path used by the web app.

For each matched `.osz`:

- unpack and parse the set
- select the top constant-BPM taiko difficulty
- reuse its audio, metadata, timing, density, difficulty, and beatmap-id conditioning
- generate one AI chart with a chosen webapp model ID
- create a cleaned reference copy with inherited timing points removed
- package both maps as neutral `A` / `B` difficulties in one shared `.osz`
- save an answer key next to the outputs without printing the mapping in the notebook


In [1]:
from __future__ import annotations

import copy
import json
import random
import sys
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile


def discover_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent]
    for candidate in candidates:
        if (candidate / "src").exists() and (candidate / "webapp" / "backend" / "models.json").exists():
            return candidate
    raise RuntimeError("Could not discover repo root from the current working directory.")


repo_root = discover_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"repo_root={repo_root}")

from src.inference.infer_from_metadata import MetadataInferenceInput, notes_baseline_from_reference, song_output_to_notes_json
from src.inference.infer_from_osz import _ensure_parsed_jsons, _resolve_osz_input_paths, select_top_difficulty_chart
from src.inference.service import GenerationService, built_in_model_registry, load_model_registry
from src.preprocessing.osutaiko_reconstructor import reconstruct_osu
from src.preprocessing.unpack_osz import unpack_osz_files

manifest_path = repo_root / "webapp" / "backend" / "models.json"
try:
    model_registry = load_model_registry(manifest_path, repo_root=repo_root)
except Exception as exc:
    print(f"[warn] could not load manifest at {manifest_path}: {exc}")
    model_registry = built_in_model_registry(repo_root)

service = GenerationService(model_registry)
available_models = [model for model in service.list_models() if model.enabled]
print("Enabled models:")
for model in available_models:
    print(f"- {model.id} | {model.architecture_name} | {model.inference_kind}")


repo_root=C:\Users\28548\PythonNotebooks\taiko-diffusion
Enabled models:
- baseline_maxopt_step_055000 | taiko_transformer | autoregressive
- baseline_maxopt_step_055000_diffusion_hybrid | taiko_diffusion_refiner | hybrid_refine
- baseline_snapshot_maxopt | taiko_transformer | autoregressive
- baseline_snapshot_maxopt_diffusion_hybrid | taiko_diffusion_refiner | hybrid_refine
- sample_large_baseline | taiko_transformer | autoregressive
- sample_large_baseline_diffusion_hybrid | taiko_diffusion_refiner | hybrid_refine
- sample_large_baseline_maxopt | taiko_transformer | autoregressive
- sample_large_baseline_maxopt_diffusion_hybrid | taiko_diffusion_refiner | hybrid_refine
- sample_large_context | taiko_context_transformer | autoregressive
- sample_large_context_diffusion_hybrid | taiko_diffusion_refiner | hybrid_refine
- sample_large_context_original | taiko_context_transformer | autoregressive
- sample_large_context_original_diffusion_hybrid | taiko_diffusion_refiner | hybrid_refine


c:\Users\28548\.conda\envs\pytorch\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\28548\.conda\envs\pytorch\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [ ]:
# Choose a webapp model id here.
# Common options right now include:
# - sample_large_context
# - sample_large_context_original
# - sample_large_baseline
# - sample_large_baseline_maxopt
# - baseline_maxopt_step_055000
# - baseline_snapshot_maxopt
# Optional diffusion-hybrid choices also exist, for example:
# - sample_large_context_diffusion_hybrid
# - sample_large_context_original_diffusion_hybrid
# - sample_large_baseline_diffusion_hybrid
# - sample_large_baseline_maxopt_diffusion_hybrid
# - baseline_maxopt_step_055000_diffusion_hybrid
# - baseline_snapshot_maxopt_diffusion_hybrid

MODEL_ID = "baseline_snapshot_maxopt"
OSZ_GLOBS = [
    "webapp/runtime/blind_eval_inputs/random5_non_snapshot/*.osz",
]
OUTPUT_ROOT = repo_root / "webapp" / "runtime" / "blind_eval"

OVERWRITE_UNPACK = False
OVERWRITE_PARSED = False
KEEP_DEBUG_JSON = True
BLIND_SEED = 42

# Optional sampling overrides. Leave as None to use model defaults.
TEMPERATURE = None
TOP_P = None
TOP_K = None
MAX_DECODE_LEN = None
DEVICE = None

print(f"MODEL_ID={MODEL_ID}")
print(f"OSZ_GLOBS={OSZ_GLOBS}")
print(f"OUTPUT_ROOT={OUTPUT_ROOT}")


MODEL_ID=baseline_snapshot_maxopt
OSZ_GLOBS=['webapp/runtime/blind_eval_inputs/random5_non_snapshot/*.osz']
OUTPUT_ROOT=C:\Users\28548\PythonNotebooks\taiko-diffusion\webapp\runtime\blind_eval


In [3]:
def write_json(path: Path, payload: dict) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    return path


def sanitize_filename_component(text: str) -> str:
    invalid = '<>:"/\\|?*'
    out = "".join("_" if ch in invalid else ch for ch in str(text))
    return out.strip().rstrip(".")


def build_standard_osu_filename(metadata_json: dict, version_label: str) -> str:
    md = dict(metadata_json.get("metadata", {}))
    artist = sanitize_filename_component(md.get("Artist", "Unknown Artist")) or "Unknown Artist"
    title = sanitize_filename_component(md.get("Title", "Untitled")) or "Untitled"
    creator = sanitize_filename_component(md.get("Creator", "Unknown")) or "Unknown"
    version = sanitize_filename_component(version_label) or "Generated"
    return f"{artist} - {title} ({creator}) [{version}].osu"


def clean_reference_timing_json(timing_json: dict) -> dict:
    cleaned = copy.deepcopy(timing_json)
    cleaned["timing_points"] = [
        copy.deepcopy(tp)
        for tp in list(timing_json.get("timing_points", []))
        if int(tp.get("uninherited", 1)) == 1
    ]
    return cleaned


def clean_tags_text(tags_text: str) -> str:
    banned = {"ai-generated", "generated", "taiko-diffusion", "ai"}
    kept = [token for token in str(tags_text or "").split() if token.strip().lower() not in banned]
    return " ".join(kept).strip()


def make_blind_metadata(source_metadata_json: dict, *, version_label: str, audio_filename: str) -> dict:
    payload = copy.deepcopy(source_metadata_json)
    payload.pop("inference", None)
    payload["source_osu"] = build_standard_osu_filename(payload, version_label)

    general = payload.setdefault("general", {})
    general["AudioFilename"] = str(audio_filename)

    md = payload.setdefault("metadata", {})
    shared_creator = str(md.get("Creator", "Unknown")).strip() or "Unknown"
    md["Creator"] = shared_creator
    md["Version"] = str(version_label)
    md["BeatmapID"] = "0"
    md["BeatmapSetID"] = "-1"
    md["Tags"] = clean_tags_text(md.get("Tags", ""))
    return payload


def choose_blind_assignment(chart_key: str, blind_seed: int) -> dict[str, str]:
    labels = ["A", "B"]
    roles = ["reference", "generated"]
    rng = random.Random(f"{blind_seed}:{chart_key}")
    rng.shuffle(roles)
    return {role: label for role, label in zip(roles, labels)}


def build_blind_eval_osz(audio_path: Path, osu_paths: list[Path], out_path: Path) -> Path:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with ZipFile(out_path, "w", compression=ZIP_DEFLATED) as archive:
        archive.write(audio_path, arcname=audio_path.name)
        for osu_path in osu_paths:
            archive.write(osu_path, arcname=osu_path.name)
    return out_path


def build_sampling_override() -> dict:
    payload = {}
    if TEMPERATURE is not None:
        payload["temperature"] = float(TEMPERATURE)
    if TOP_P is not None:
        payload["top_p"] = float(TOP_P)
    if TOP_K is not None:
        payload["top_k"] = int(TOP_K)
    if MAX_DECODE_LEN is not None:
        payload["max_decode_len"] = int(MAX_DECODE_LEN)
    if DEVICE is not None:
        payload["device"] = str(DEVICE)
    return payload


def resolve_notebook_osz_inputs(osz_inputs: list[str | Path], *, repo_root: Path) -> list[str]:
    normalized_inputs = []
    for raw_input in osz_inputs:
        text = str(raw_input)
        path = Path(text)
        if path.is_absolute():
            normalized_inputs.append(str(path))
            continue
        normalized_inputs.append(str((repo_root / path).resolve()))
    return _resolve_osz_input_paths(normalized_inputs)


def run_blind_eval_for_chart(chart, *, model_id: str, output_root: Path, blind_seed: int, keep_debug_json: bool) -> dict:
    model = service.get_model(model_id)
    chart_key = f"{chart.unpacked_dir.name}:{chart.stem}"
    chart_output_dir = Path(output_root) / chart.unpacked_dir.name / chart.stem
    chart_output_dir.mkdir(parents=True, exist_ok=True)

    cleaned_timing_json = clean_reference_timing_json(chart.timing_json)
    cleaned_timing_path = write_json(chart_output_dir / f"{chart.stem}.cleaned.timing.json", cleaned_timing_json)

    chart_input = MetadataInferenceInput(
        chart_stem=chart.stem,
        audio_path=chart.audio_path,
        metadata_json=chart.metadata_json,
        timing_json=chart.timing_json,
        offset_ms=chart.offset_ms,
        bpm=chart.bpm,
        meter=chart.meter,
        difficulty_value=chart.difficulty_value,
        beatmap_id=chart.beatmap_id,
        density_nps=chart.density_nps,
        reference_notes_json=chart.notes_json,
    )
    song_output, architecture_spec = service._generate_song_output_for_model(
        model,
        chart_input=chart_input,
        sampling_override=build_sampling_override(),
    )

    sv_default, volume_default = notes_baseline_from_reference(chart.notes_json)
    generated_notes_json = song_output_to_notes_json(
        song_output,
        source_osu=str(chart.metadata_json.get("source_osu", f"{chart.stem}.osu")),
        offset_ms=chart.offset_ms,
        bpm=chart.bpm,
        meter=chart.meter,
        sv_default=sv_default,
        volume_default=volume_default,
    )
    generated_notes_path = write_json(chart_output_dir / f"{chart.stem}.generated.notes.json", generated_notes_json)

    assignment = choose_blind_assignment(chart_key, blind_seed)
    reference_label = assignment["reference"]
    generated_label = assignment["generated"]

    blind_reference_metadata = make_blind_metadata(
        chart.metadata_json,
        version_label=reference_label,
        audio_filename=chart.audio_path.name,
    )
    blind_generated_metadata = make_blind_metadata(
        chart.metadata_json,
        version_label=generated_label,
        audio_filename=chart.audio_path.name,
    )
    blind_reference_metadata_path = write_json(
        chart_output_dir / f"{chart.stem}.reference.{reference_label}.metadata.json",
        blind_reference_metadata,
    )
    blind_generated_metadata_path = write_json(
        chart_output_dir / f"{chart.stem}.generated.{generated_label}.metadata.json",
        blind_generated_metadata,
    )

    reference_osu_path = chart_output_dir / build_standard_osu_filename(blind_reference_metadata, reference_label)
    generated_osu_path = chart_output_dir / build_standard_osu_filename(blind_generated_metadata, generated_label)

    reconstruct_osu(
        notes_path=chart.notes_path,
        out_path=reference_osu_path,
        timing_path=cleaned_timing_path,
        metadata_path=blind_reference_metadata_path,
    )
    reconstruct_osu(
        notes_path=generated_notes_path,
        out_path=generated_osu_path,
        timing_path=cleaned_timing_path,
        metadata_path=blind_generated_metadata_path,
    )

    if keep_debug_json:
        write_json(chart_output_dir / f"{chart.stem}.original.timing.json", chart.timing_json)
        write_json(chart_output_dir / f"{chart.stem}.song_output.json", {"song_output": song_output})

    package_name = f"{sanitize_filename_component(chart.metadata_json.get('metadata', {}).get('Artist', 'Unknown Artist'))} - {sanitize_filename_component(chart.metadata_json.get('metadata', {}).get('Title', 'Untitled'))} - blind-eval.osz"
    blind_osz_path = build_blind_eval_osz(
        chart.audio_path,
        [reference_osu_path, generated_osu_path],
        chart_output_dir / package_name,
    )

    answer_key = {
        "source_osz": chart.unpacked_dir.name,
        "chart_stem": chart.stem,
        "model_id": model_id,
        "architecture_name": str(architecture_spec.name),
        "reference_label": reference_label,
        "generated_label": generated_label,
        "reference_osu": reference_osu_path.name,
        "generated_osu": generated_osu_path.name,
    }
    answer_key_path = write_json(chart_output_dir / "answer_key.json", answer_key)

    return {
        "source_osz": chart.unpacked_dir.name,
        "chart_stem": chart.stem,
        "model_id": model_id,
        "output_dir": chart_output_dir,
        "blind_osz_path": blind_osz_path,
        "answer_key_path": answer_key_path,
    }


In [4]:
resolved_osz_inputs = resolve_notebook_osz_inputs(OSZ_GLOBS, repo_root=repo_root)
work_root = Path(OUTPUT_ROOT) / "_work"
unpacked_root = work_root / "unpacked"

print("Matched .osz inputs:")
for path in resolved_osz_inputs:
    print(f"- {path}")

unpacked_dirs = unpack_osz_files(
    source_paths=resolved_osz_inputs,
    destination_root=unpacked_root,
    overwrite=bool(OVERWRITE_UNPACK),
    keep_only_chart_and_audio=True,
    progress_desc="Unpacking blind-eval .osz files",
)

selected_charts = []
for unpacked_dir in unpacked_dirs:
    unpacked_dir = Path(unpacked_dir).resolve()
    _ensure_parsed_jsons(unpacked_dir, overwrite_parsed=bool(OVERWRITE_PARSED))
    chart = select_top_difficulty_chart(unpacked_dir)
    selected_charts.append(chart)
    print(
        f"[select] {unpacked_dir.name} -> {chart.stem} | "
        f"OD={chart.overall_difficulty:.2f} notes={chart.playable_notes} "
        f"density={chart.density_nps:.2f} bpm={chart.bpm:.3f}"
    )

results = []
for chart in selected_charts:
    result = run_blind_eval_for_chart(
        chart,
        model_id=MODEL_ID,
        output_root=Path(OUTPUT_ROOT),
        blind_seed=int(BLIND_SEED),
        keep_debug_json=bool(KEEP_DEBUG_JSON),
    )
    results.append(result)
    print(f"[done] blind package -> {result['blind_osz_path']}")

print(f"Created {len(results)} blind-eval package(s).")


FileNotFoundError: No .osz files matched the provided --osz-inputs.

In [ ]:
print("Blind-eval outputs:")
for result in results:
    print(f"- source_osz={result['source_osz']}")
    print(f"  chart_stem={result['chart_stem']}")
    print(f"  blind_osz={result['blind_osz_path']}")
    print(f"  output_dir={result['output_dir']}")
    print(f"  answer_key_saved_to={result['answer_key_path']}")
    print("  note: the answer-key mapping is saved to disk and not printed here.")
